In [37]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [38]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")


In [39]:
subgraphLLM = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    google_api_key=api_key,
)

In [40]:
class SubState(TypedDict):
    input_str:str
    trans_str:str

In [41]:
def translate_text(state:SubState):
       prompt = f"""
Translate the following text to Urdu.
keep the text as it is dont add by yourself

Text:
{state["input_str"]}
""".strip()
       res=subgraphLLM.invoke(prompt).content
       return {'trans_str':res}


In [42]:
subgraph=StateGraph(SubState)

subgraph.add_node('translate',translate_text)

subgraph.add_edge(START,'translate')
subgraph.add_edge('translate',END)

subgraphWorkflow=subgraph.compile()

In [43]:
class ParentState(TypedDict):
    user_input:str
    gen_text:str
    trans_text:str
    

In [44]:
llm = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    google_api_key=api_key,
)

In [45]:
def generate_answer(state:ParentState):
    prompt=f"Generate a text on topic {state['user_input']}"

    res=llm.invoke(prompt).content
    return {'gen_text':res}
    

In [46]:
def translate_answer(state:ParentState):
    res=subgraphWorkflow.invoke({'input_str':state['gen_text']})

    return {'trans_text':res['trans_str']}

In [47]:
graph=StateGraph(ParentState)

graph.add_node('generate_answer',generate_answer)
graph.add_node('translate_answer',translate_answer)

graph.add_edge(START,'generate_answer')
graph.add_edge('generate_answer','translate_answer')
graph.add_edge('translate_answer',END)

workflow=graph.compile()


In [48]:
workflow.invoke({'user_input':"Ai"})

{'user_input': 'Ai',
 'gen_text': '## The Dawn of Intelligence: Exploring the Landscape of AI\n\nArtificial Intelligence (AI) is no longer a concept confined to the realm of science fiction; it is a rapidly evolving reality profoundly reshaping our world. At its core, AI refers to the development of computer systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, perception, and understanding language.\n\nThe journey of AI began with symbolic logic and rule-based systems, but its recent explosion in capability is largely attributed to advancements in **Machine Learning (ML)** and **Deep Learning (DL)**. These subfields enable AI systems to learn from vast amounts of data, identify patterns, and make predictions or decisions without explicit programming for every scenario. Neural networks, inspired by the human brain\'s structure, are the backbone of many modern AI applications, allowing them to process complex informa